In [ ]:
# Versi dengan Masking layer — NON-cuDNN (lebih lambat, tapi masking berjalan benar)
# recurrent_dropout=1e-7 menonaktifkan cuDNN kernel sehingga Masking pada
# Bidirectional LSTM tidak menyebabkan error reversed-mask dari arah mundur.

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout, Masking,
    Lambda, Subtract, Multiply, concatenate
)
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# Kaggle  : set USE_AUGMENTED = True/False sesuai kebutuhan
# Lokal   : ubah DATA_DIR ke path lokal

DATASET_SLUG  = "siamese-data"   # <-- GANTI sesuai nama dataset Kaggle
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True             # True  → pakai aug_*.npy + aug_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'aug_' if USE_AUGMENTED else ''
meta_f = 'aug_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# ── Hyperparameter (samakan gaya dengan `siamese_bilstm_direct.ipynb`) ─────────
BILSTM_UNITS = 128
DROPOUT      = 0.3
EPOCHS       = 100
BATCH_SIZE   = 32
PATIENCE     = 10


def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                bilstm_units=128, dropout=0.3):
    """
    Siamese BiLSTM direct regression (grade 1-10) DENGAN Masking + non-cuDNN.
    recurrent_dropout=1e-7 menonaktifkan cuDNN agar Masking berjalan benar
    pada Bidirectional LSTM.

    Encoder:
      - bilstm_question : khusus question
      - shared_bilstm   : dipakai answerkey dan answer (Siamese)

    Fitur: [eq, ea, eak, |eak-ea|, eak⊙ea]  →  5 × 256D = 1280D
    Output: Dense(1, linear)
    """
    bilstm_q = Bidirectional(
        LSTM(bilstm_units, return_sequences=False, recurrent_dropout=1e-7),
        name='bilstm_question'
    )
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=False, recurrent_dropout=1e-7),
        name='shared_bilstm'
    )

    inp_q  = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a  = Input(shape=(a_seq_len,  emb_dim), name='inp_a')

    eq  = bilstm_q(Masking(mask_value=0.0)(inp_q))
    eak = shared_bilstm(Masking(mask_value=0.0)(inp_ak))
    ea  = shared_bilstm(Masking(mask_value=0.0)(inp_a))

    diff     = Subtract(name='diff')([eak, ea])
    abs_diff = Lambda(lambda x: tf.abs(x), name='abs_diff')(diff)
    had_prod = Multiply(name='had_prod')([eak, ea])

    merged = concatenate([eq, ea, eak, abs_diff, had_prod], name='merged')

    x   = Dense(256, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(64,  activation='relu')(x)
    out = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out,
                  name='siamese_bilstm_direct_nomask')

    huber_loss = Huber(delta=1.0)
    model.compile(optimizer='adam', loss=huber_loss, metrics=['mae', 'mse'])
    return model


# Preview
_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2]
)
_tmp.summary()
del _tmp


In [ ]:
# ── LOPO Cross-Validation (Leave-One-Participant-Out) ─────────────────────────
# Samakan dengan `siamese_bilstm_direct.ipynb`:
# - Augmentasi hanya di train
# - Val & Test hanya data asli
# - Prediksi grade langsung + hitung RMSE/MAE

idpsj_list = sorted(metadata['IDPSJ'].unique())
n_parts    = len(idpsj_list)
y_all      = metadata['grade'].values.astype(np.float32)
is_real    = ~metadata['IDJwb'].astype(str).str.startswith('syn_')

fold_results = []

for i, test_id in enumerate(idpsj_list):
    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test={test_id}  |  Val={val_id}")

    train_mask = metadata['IDPSJ'].isin(train_ids)
    train_idx  = metadata.index[train_mask].values
    val_idx    = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    test_idx   = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    y_test  = y_all[test_idx]

    print(f"  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_tr  = get_split(questions_emb,  train_idx)
    X_ak_tr = get_split(answerkeys_emb, train_idx)
    X_a_tr  = get_split(answers_emb,    train_idx)

    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)

    X_q_te  = get_split(questions_emb,  test_idx)
    X_ak_te = get_split(answerkeys_emb, test_idx)
    X_a_te  = get_split(answers_emb,    test_idx)

    model = build_model(
        q_seq_len    = questions_emb.shape[1],
        ak_seq_len   = answerkeys_emb.shape[1],
        a_seq_len    = answers_emb.shape[1],
        emb_dim      = answers_emb.shape[2],
        bilstm_units = BILSTM_UNITS,
        dropout      = DROPOUT,
    )

    # ── Sample weights (inverse frequency per grade, per fold) ───────────────
    grade_int = y_train.astype(int)
    unique_g, counts_g = np.unique(grade_int, return_counts=True)
    freq_map  = dict(zip(unique_g, counts_g))
    n_kelas   = len(unique_g)
    raw_w     = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
    sample_w  = raw_w / raw_w.mean()
    print(f"  Sample weight  min={sample_w.min():.2f}  max={sample_w.max():.2f}  mean={sample_w.mean():.2f}")

    reduce_lr = ReduceLROnPlateau(
        monitor='val_mae',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )

    early_stop = EarlyStopping(
        monitor='val_mae',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    )

    print("  Memulai proses training...")
    model.fit(
        [X_q_tr, X_ak_tr, X_a_tr], y_train,
        sample_weight=sample_w,
        validation_data=([X_q_val, X_ak_val, X_a_val], y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[reduce_lr, early_stop],
        verbose=1
    )

    y_pred_raw = model.predict([X_q_te, X_ak_te, X_a_te], verbose=0).flatten()
    y_pred     = np.clip(np.round(y_pred_raw), 1, 10)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    print(f"  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}")

    fold_results.append({
        'fold'       : i + 1,
        'test_idpsj' : test_id,
        'val_idpsj'  : val_id,
        'n_train'    : len(y_train),
        'n_val'      : len(y_val),
        'n_test'     : len(y_test),
        'rmse'       : rmse,
        'mae'        : mae,
        'y_test'     : y_test,
        'y_pred'     : y_pred,
    })

    model_path = os.path.join(OUT_DIR, f'nomask_model_fold_{i+1:02d}.keras')
    model.save(model_path)
    print(f"  Model saved -> {model_path}")

    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")

In [ ]:
# ── Evaluasi Akhir (samakan dengan direct) ───────────────────────────────────

summary = pd.DataFrame([{
    'fold'       : r['fold'],
    'test_idpsj' : r['test_idpsj'],
    'n_train'    : r['n_train'],
    'n_test'     : r['n_test'],
    'RMSE'       : round(r['rmse'], 4),
    'MAE'        : round(r['mae'],  4),
} for r in fold_results])

print("=" * 60)
print("Hasil per Fold (NoMask Direct)")
print("=" * 60)
print(summary.to_string(index=False))
print(f"\nRata-rata  RMSE : {summary['RMSE'].mean():.4f} ± {summary['RMSE'].std():.4f}")
print(f"Rata-rata  MAE  : {summary['MAE'].mean():.4f}  ± {summary['MAE'].std():.4f}")

# Simpan ringkasan
summary.to_csv(os.path.join(OUT_DIR, 'nomask_lopo_results.csv'), index=False)

# Plot RMSE & MAE per fold
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['RMSE', 'MAE']):
    ax.bar(summary['test_idpsj'].astype(str), summary[metric], color='steelblue', edgecolor='black')
    ax.axhline(summary[metric].mean(), color='red', linestyle='--', label=f'Mean {metric}')
    ax.set_title(f'{metric} per Fold (test IDPSJ)')
    ax.set_xlabel('Test IDPSJ')
    ax.set_ylabel(metric)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'nomask_lopo_metrics.png'), dpi=150)
plt.show()

# Scatter: prediksi vs aktual (gabungan semua fold)
y_all_true = np.concatenate([r['y_test'] for r in fold_results])
y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])

plt.figure(figsize=(6, 6))
plt.scatter(y_all_true, y_all_pred, alpha=0.5, edgecolors='k', linewidths=0.3)
plt.plot([1, 10], [1, 10], 'r--', label='Ideal')
plt.xlabel('Grade Aktual')
plt.ylabel('Grade Prediksi')
plt.title('Prediksi vs Aktual (semua fold)')
plt.xticks(range(1, 11))
plt.yticks(range(1, 11))
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'nomask_scatter_all_folds.png'), dpi=150)
plt.show()